# PyTorch 基础CNN核心知识点讲解
## 一、全连接神经网络（FCN）回顾
### 1. 输入输出维度变换
以MNIST数据集为例，输入为28×28的灰度图，维度变换流程：
$(N, 1, 28, 28) \rightarrow (N, 784) \rightarrow (N, 512) \rightarrow (N, 256) \rightarrow (N, 128) \rightarrow (N, 64) \rightarrow (N, 10)$
- 先将二维图像展平为一维向量（784=28×28）
- 经多层线性层+ReLU激活，最终输出10类分类结果

### 2. PyTorch实现代码
```python
import torch
import torch.nn.functional as F

class Net(torch.nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.l1 = torch.nn.Linear(784, 512)
        self.l2 = torch.nn.Linear(512, 256)
        self.l3 = torch.nn.Linear(256, 128)
        self.l4 = torch.nn.Linear(128, 64)
        self.l5 = torch.nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(-1, 784)  # 展平
        x = F.relu(self.l1(x))
        x = F.relu(self.l2(x))
        x = F.relu(self.l3(x))
        x = F.relu(self.l4(x))
        return self.l5(x)

model = Net()
```

### 3. 全连接网络的问题
丢失图像的**空间结构信息**，参数数量庞大，易过拟合，对图像类任务效果不佳。

## 二、卷积神经网络（CNN）核心组件
CNN核心分为**特征提取**（卷积+池化）和**分类**（全连接）两部分，保留图像空间特征，参数共享大幅减少计算量。

### 1. 卷积层（Conv2d）
#### （1）卷积的基本原理
通过**卷积核（Kernel/Filter）** 在输入特征图上滑动，逐元素相乘再求和，提取局部特征，分为单输入通道和多输入通道卷积。
- **单通道卷积**：输入为单通道特征图，单个卷积核完成计算，如5×5输入+3×3卷积核，输出3×3特征图
- **多通道卷积**：输入为$n$通道，需$n$个同尺寸卷积核，各通道卷积结果相加得到单通道输出；若要输出$m$通道，需$m$组（每组$n$个）卷积核

#### （2）卷积核张量形状
卷积层的权重张量为**4维**：$out\_channels \times in\_channels \times kernel\_size_w \times kernel\_size_h$
- 例：输入5通道，输出10通道，3×3卷积核 → 权重形状$(10,5,3,3)$

#### （3）关键参数
- **padding**：在输入特征图边缘补0，保持输出维度与输入一致，`padding=1`表示边缘补1圈0
- **stride**：卷积核滑动的步长，`stride=2`表示每次滑动2个像素，输出维度会缩小
- **bias**：偏置项，可选是否添加

#### （4）输出维度计算公式
**3/2=1,5x5矩阵减一圈得到3x3矩阵**

默认`dilation=1`（无空洞卷积）时：
$$out\_size = \lfloor \frac{in\_size - kernel\_size + 2 \times padding}{stride} \rfloor + 1$$
- 例：5×5输入，3×3卷积核，`padding=0`，`stride=1` → 输出3×3。
- 例：5×5输入，3×3卷积核，`padding=1`，`stride=1` → 输出5×5
- 例：5×5输入，3×3卷积核，`padding=0`，`stride=2` → 输出2×2

#### （5）PyTorch实现示例
```python
import torch
# 构造输入：batch_size=1, in_channels=1, 5×5
input = torch.Tensor([3,4,6,5,7,2,4,6,8,2,1,6,7,8,4,9,7,4,6,2,3,7,5,4,1]).view(1,1,5,5)
# 定义卷积层：1输入通道，1输出通道，3×3卷积核，padding=1，无偏置
conv_layer = torch.nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=False)
# 自定义卷积核
kernel = torch.Tensor([1,2,3,4,5,6,7,8,9]).view(1,1,3,3)
conv_layer.weight.data = kernel.data
# 前向传播
output = conv_layer(input)
print(output)
```

### 2. 池化层（Pooling）
#### （1）最大池化（MaxPool2d）
取卷积核覆盖区域的最大值，**下采样**缩小特征图，保留关键特征，降低计算量，防止过拟合。
- 常用2×2池化核，stride=2，将特征图尺寸缩小为原来的1/2

#### （2）PyTorch实现示例
```python
import torch
# 构造4×4输入
input = torch.Tensor([3,4,6,5,2,4,6,8,1,6,7,8,9,7,4,6]).view(1,1,4,4)
# 2×2最大池化
maxpooling_layer = torch.nn.MaxPool2d(kernel_size=2)
output = maxpooling_layer(input)
print(output)  # 输出2×2特征图
```

#### （3）池化层的特点
无可训练参数，仅做特征筛选和维度压缩。

### 3. 激活层
与全连接网络一致，常用**ReLU**，在卷积/池化后添加，引入非线性，提升网络表达能力，使用`torch.nn.functional.relu()`实现。

### 4. 全连接层（Linear）
将卷积+池化提取的高维特征图**展平**为一维向量，经多层线性层完成分类/回归任务，是CNN的分类头。

## 三、基础CNN网络搭建（MNIST为例）
### 1. 网络结构与维度变换
输入：$(batch, 1, 28, 28)$（MNIST灰度图）
1. Conv2d(1,10,5) → $(batch, 10, 24, 24)$ → ReLU → MaxPool2d(2) → $(batch, 10, 12, 12)$
2. Conv2d(10,20,5) → $(batch, 20, 8, 8)$ → ReLU → MaxPool2d(2) → $(batch, 20, 4, 4)$
3. 展平：$(batch, 20×4×4) = (batch, 320)$
4. Linear(320,10) → $(batch, 10)$（最终10类输出）

### 2. PyTorch完整实现代码
```python
import torch
import torch.nn.functional as F

class Net(torch.nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # 卷积层1：1输入通道，10输出通道，5×5卷积核
        self.conv1 = torch.nn.Conv2d(1, 10, kernel_size=5)
        # 卷积层2：10输入通道，20输出通道，5×5卷积核
        self.conv2 = torch.nn.Conv2d(10, 20, kernel_size=5)
        # 2×2最大池化层（共享）
        self.pooling = torch.nn.MaxPool2d(2)
        # 全连接层：320输入特征，10输出类别
        self.fc = torch.nn.Linear(320, 10)

    def forward(self, x):
        batch_size = x.size(0)  # 获取批次大小
        # 卷积1 + 激活 + 池化
        x = F.relu(self.pooling(self.conv1(x)))
        # 卷积2 + 激活 + 池化
        x = F.relu(self.pooling(self.conv2(x)))
        # 展平特征图
        x = x.view(batch_size, -1)
        # 全连接层输出
        x = self.fc(x)
        return x

# 实例化模型
model = Net()
```

## 四、CNN的GPU加速
PyTorch中需将**模型**和**数据**同时移至GPU，才能实现加速，核心步骤如下：
### 1. 设备选择
```python
# 检测是否有可用GPU，无则使用CPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
```

### 2. 模型移至GPU
```python
model = Net()
model.to(device)  # 将模型的参数和缓冲区转为CUDA张量
```

### 3. 数据移至GPU
训练和测试时，每次迭代都需将输入和标签移至GPU：
#### （1）训练函数
```python
def train(epoch, train_loader, model, criterion, optimizer, device):
    running_loss = 0.0
    for batch_idx, data in enumerate(train_loader, 0):
        inputs, target = data
        # 数据移至GPU
        inputs, target = inputs.to(device), target.to(device)
        # 梯度清零
        optimizer.zero_grad()
        # 前向传播
        outputs = model(inputs)
        # 计算损失
        loss = criterion(outputs, target)
        # 反向传播+参数更新
        loss.backward()
        optimizer.step()
        # 累计损失
        running_loss += loss.item()
        # 打印损失
        if batch_idx % 300 == 299:
            print('[%d, %5d] loss: %.3f' % (epoch + 1, batch_idx + 1, running_loss / 300))
            running_loss = 0.0
```

#### （2）测试函数
```python
def test(test_loader, model, device):
    correct = 0
    total = 0
    with torch.no_grad():  # 关闭梯度计算
        for data in test_loader:
            inputs, target = data
            # 数据移至GPU
            inputs, target = inputs.to(device), target.to(device)
            # 前向传播
            outputs = model(inputs)
            # 获取预测类别
            _, predicted = torch.max(outputs.data, dim=1)
            total += target.size(0)
            # 统计正确数
            correct += (predicted == target).sum().item()
    # 打印测试准确率
    print('Accuracy on test set: %d %% [%d/%d]' % (100 * correct / total, correct, total))
```

### 4. 加速效果
MNIST数据集上，GPU训练的基础CNN迭代10轮后，**测试准确率可达98%以上**，远高于全连接网络，且训练速度大幅提升。

## 五、CNN的扩展与优化
### 1. 更复杂的CNN结构设计
可通过增加卷积层、池化层、全连接层的数量提升模型表达能力，例如：
- 3层Conv2d + 3层ReLU + 3层MaxPooling + 3层Linear
- 调整卷积核大小（3×3/5×5）、输出通道数、padding/stride参数

### 2. 超参数调优
- 学习率：选择合适的学习率（如0.001/0.01），可使用学习率衰减
- 批次大小（batch_size）：平衡训练速度和模型收敛性
- 正则化：添加Dropout、L2正则化防止过拟合
- 优化器：使用Adam、SGD+动量等优化器提升训练效率

### 3. 核心优化原则
- 卷积层：逐步增加输出通道数，提取更复杂的特征
- 池化层：在卷积层后添加，逐步缩小特征图尺寸
- 全连接层：尽量减少层数和神经元数，避免参数过多

## 六、CNN核心总结
1. **核心优势**：保留图像空间结构，参数共享，计算量小，泛化能力强
2. **核心组件**：卷积层（特征提取）、池化层（下采样）、激活层（非线性）、全连接层（分类）
3. **维度把控**：卷积/池化的`padding`和`stride`是控制特征图维度的关键，需熟练掌握输出维度计算公式
4. **GPU加速**：模型和数据必须同时移至GPU，缺一不可
5. **设计思路**：先通过卷积+池化提取多层特征，再通过全连接层完成分类/回归，遵循**特征提取→特征压缩→分类**的逻辑


In [ ]:

import torch
from torchvision import datasets,transforms
from torch.utils.data import DataLoader
from torch.nn import functional as F
batch_size = 64
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data/mnist/', train=True, download=True,transform=transform)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_dataset = datasets.MNIST(root='./data/mnist/', train=False, download=True,transform=transform)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

class Net(torch.nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = torch.nn.Conv2d(1, 10, kernel_size=5)
        self.pool = torch.nn.MaxPool2d(2)
        self.conv2 = torch.nn.Conv2d(10, 20, kernel_size=5)
        self.fc1 = torch.nn.Linear(20 * 4 * 4, 128)
        self.fc2 = torch.nn.Linear(128, 10)
    def forward(self, x):
        batch_size = x.size(0)
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(batch_size, -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = Net()
device = torch.device("xpu" if torch.xpu.is_available() else "cpu")
model.to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.5)

def train(epoch):
    running_loss = 0.0
    model.train()
    for batch_idx, (inputs, target) in enumerate(train_loader):
        inputs,target = inputs.to(device),target.to(device)
        optimizer.zero_grad()
        output = model(inputs)
        loss = criterion(output, target)
        loss.backward()
        
        optimizer.step()
        running_loss += loss.item()
        
        if batch_idx % 300 == 299:
            print(f"[{epoch+1}, {batch_idx+1}] loss: {running_loss/300:.3f}")
            running_loss = 0.0
   
def test():
    correct = 0
    total = 0
    model.eval()
    with torch.no_grad():
        for data in test_loader:
            inputs, target = data
            inputs,target = inputs.to(device),target.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    accuracy = 100 * correct / total
    print(f'Accuracy on test set: {accuracy:.2f} %[{correct}/{total}]')
    # return accuracy  # 返回准确率，便于监控




In [ ]:
if __name__ == '__main__':
    for epoch in range(5):  # 训练30个epoch
        train(epoch)
        test()

In [2]:

import torch
from torchvision import datasets,transforms
from torch.utils.data import DataLoader
from torch.nn import functional as F
batch_size = 64
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data/mnist/', train=True, download=True,transform=transform)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_dataset = datasets.MNIST(root='./data/mnist/', train=False, download=True,transform=transform)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

class Net(torch.nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = torch.nn.Conv2d(1, 10, kernel_size=5,padding=1) #1,28,28->10,24,24->10,12,12
        self.conv2 = torch.nn.Conv2d(10, 20, kernel_size=3,padding=1) #->20,12,12->20,6,6
        self.conv3 = torch.nn.Conv2d(20, 30, kernel_size=3, padding=1) #->30,6,6->30,3,3
        self.pool = torch.nn.MaxPool2d(2)
        self.fc1 = torch.nn.Linear(30*3*3, 128) #->128
        self.fc2 = torch.nn.Linear(128, 10)
    def forward(self, x):
        batch_size = x.size(0)
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        # temp = x.shape
        # print(temp)
        x = x.view(batch_size, -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = Net()
device = torch.device("xpu" if torch.xpu.is_available() else "cpu")
model.to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.5)

def train(epoch):
    running_loss = 0.0
    model.train()
    for batch_idx, (inputs, target) in enumerate(train_loader):
        inputs,target = inputs.to(device),target.to(device)
        optimizer.zero_grad()
        output = model(inputs)
        loss = criterion(output, target)
        loss.backward()
        
        optimizer.step()
        running_loss += loss.item()
        
        if batch_idx % 300 == 299:
            print(f"[{epoch+1}, {batch_idx+1}] loss: {running_loss/300:.3f}")
            running_loss = 0.0
   
def test():
    correct = 0
    total = 0
    model.eval()
    with torch.no_grad():
        for data in test_loader:
            inputs, target = data
            inputs,target = inputs.to(device),target.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    accuracy = 100 * correct / total
    print(f'Accuracy on test set: {accuracy:.2f} %[{correct}/{total}]')
    # return accuracy  # 返回准确率，便于监控

if __name__ == '__main__':
    for epoch in range(5):  # 训练30个epoch
        train(epoch)
        test()


[1, 300] loss: 1.571
[1, 600] loss: 0.308
[1, 900] loss: 0.185
Accuracy on test set: 95.14 %[9514/10000]
[2, 300] loss: 0.146
[2, 600] loss: 0.127
[2, 900] loss: 0.115
Accuracy on test set: 97.17 %[9717/10000]
[3, 300] loss: 0.095
[3, 600] loss: 0.091
[3, 900] loss: 0.089
Accuracy on test set: 97.68 %[9768/10000]
[4, 300] loss: 0.075
[4, 600] loss: 0.074
[4, 900] loss: 0.069
Accuracy on test set: 98.16 %[9816/10000]
[5, 300] loss: 0.061
[5, 600] loss: 0.060
[5, 900] loss: 0.061
Accuracy on test set: 98.57 %[9857/10000]
